# 3. Keyword methods: M0, M1, M1n, M1p

**Reads:** `intermediate/sentences.pkl`, `intermediate/gold_sentences.csv`.
**Writes:** predictions for M0, M1, M1n and M1p in `predictions/`, and `intermediate/m1_hits.csv` (the phrases each M1 hit matched, shown in the error review in `06_method_comparison`).

## Setup

In [1]:
from argmine_common import *

NEGATION_WINDOW = 6            # words before a keyword checked for negation

sents = load_sentences()
GOLD_IDX, Y, DOCS, gold_docs, TEST = load_gold()

## M0: original keyword lists (baseline)

The dictionary and matching logic from `nidhi_working.ipynb`, unchanged, including the AND rule for C2 and the misspelled "vexacious lawsuit". The only difference is the unit: it now runs on sentences, so every method is scored on identical units.

In [2]:
NIDHI_KEYWORDS = {
    "C1": ['elements of the offence', "elements of the offense", "ingredients of the offense",
           "ingredients of the offence", 'exception', 'justification', 'mens rea', 'fair comment',
           'good faith', "section 499"],
    "C2": {"speech_keywords": ['freedom of speech', 'freedom of expression', 'freedom of press',
                               'article 19(1)(a)', 'constitutional right', 'right to speak',
                               'right to criticize', 'right to criticism', 'public criticism',
                               'political criticism', 'fair criticism', 'criticism of public officials',
                               'criticism of public figures', 'public interest'],
           "reputation_keywords": ['reputation', 'dignity', 'impact on reputation', 'harm to reputation',
                                   'defamatory', 'good reputation']},
    "C3": ['jurisdiction', 'procedural', 'delay', 'limitation', 'maintainability', 'admissibility',
           'pleading', 'summons', 'notice', 'filing', 'technicality', 'irregularity',
           'cancellation of bail', 'vitiate'],
    "C4": ['no evidence', 'no proof', 'insufficient evidence', 'insufficient proof', 'lack of evidence',
           'lack of proof', 'failed to prove', 'failed to establish', 'lack of corroboration',
           'no corroboration', 'not established', 'insufficient testimony', 'without specific evidence',
           'failed to provide', 'could not establish'],
    "C5": ['abuse of process', 'misuse of process', 'abuse of power', 'misuse of power', 'vexacious lawsuit'],
}

def _present(text, phrase):
    return bool(re.search(r"\b" + re.escape(phrase.lower().strip()) + r"\b", text))

def m0_row(text):
    t = str(text).lower()
    out = []
    for c in CATS:
        cfg = NIDHI_KEYWORDS[c]
        if isinstance(cfg, dict):
            hit = (any(_present(t, k) for k in cfg["speech_keywords"])
                   and any(_present(t, k) for k in cfg["reputation_keywords"]))
        else:
            hit = any(_present(t, k) for k in cfg)
        out.append(int(hit))
    return out

M0_ALL = np.array([m0_row(t) for t in sents.text])
save_preds("M0 original keywords", M0_ALL[GOLD_IDX], M0_ALL)
per_category(Y, M0_ALL[GOLD_IDX]).round(2)

,precision,recall,f1,support
Ingredients/exceptions,0.44,0.33,0.38,51
Speech vs reputation,0.00,0.00,0.00,5
Procedural,0.11,0.16,0.13,31
Insufficient evidence,0.20,0.04,0.06,27
Abuse of process,1.00,0.14,0.24,29


## M1: revised phrase lists, citations, scope blockers, negation

Changes from M0, each addressing a failure seen in the old outputs:
* **Generic terms removed** (`notice`, `filing`, `delay`, `section 499`, `defamatory`), since they appear in nearly every defamation judgment.
* **Paraphrases of abuse of process added** ("mala fide", "counter blast", "wreak vengeance", "ulterior motive"), and the "vexatious" typo fixed.
* **Precedent citations** as category signals. *The citation list needs the lawyers' sign off.*
* **Scope blockers** discard a match in a known wrong context: "inherent jurisdiction" is the court's quashing power, not a jurisdictional defect; "no evidence of malice" concerns the intent ingredient (C1), not evidential insufficiency (C4); "sexual harassment" is not harassment of the accused.
* **Procedural defects, not procedural description** (second revision, from the dev highlights). Lawyers mark C3 when the judge finds a step was done wrongly (no jurisdiction to look at the accused's documents, an inadmissible document, a case remanded for a fresh hearing), not when the judge explains the court's powers or quotes Section 202. Bare "jurisdiction", bare section numbers (190, 199, 200, 202, ...) and "at this stage" are replaced by defect phrasings ("no jurisdiction", "exceeded its jurisdiction", "without holding an inquiry", "inadmissible", "locus standi", "remanded", "afresh"). C4 adds absence of evidence and burden of proof phrasings; C2 adds decriminalisation. These patterns were chosen by reading dev sentences, so M1's dev score is optimistic; its test score is not.
* **Negation** is recorded for every hit ("not an abuse of process"). M1 keeps negated hits, because rejecting a ground is still reasoning about that ground. M1n discards them. Comparing the two on the gold data tells us which reading the lawyers' annotations follow.
* **Party arguments.** Judgments restate each side's submissions before giving reasons, and those restatements contain the same keywords. M1p skips sentences flagged `party_argument`. In the *Sanwaria* annotations, several keyword hits that the lawyer did not highlight were counsel's submissions, so this is worth testing on the full gold set.

In [3]:
KEYWORD_PATTERNS = {
    "C1": [r"\bingredients?\b", r"\belements? of (the )?(offen[cs]e|defamation)\b", r"\bexceptions?\b",
           r"\bpublic good\b", r"\bgood faith\b", r"\bbona fides?\b", r"\bmens rea\b", r"\bfair comment\b",
           r"\bimputation of truth\b", r"\btruth of the (imputation|allegation|statement)s?\b",
           r"\bintend(ed|ing)? to harm\b", r"\bintention to (harm|defame|lower)\b",
           r"\bknowing or having reason to believe\b",
           r"\b(published|publication|communicated|conveyed) to (a |any |the )?third (person|party|parties)\b",
           r"\bthird (person|party|parties)\b", r"\bjuristic person\b", r"\bvicarious(ly)? liab",
           r"\bpress and registration of books act\b", r"\bjustification\b"],
    "C2": [r"\bfreedom of (speech|expression|(the )?press)\b", r"\bfree (speech|press)\b",
           r"\bright to (free )?(speech|expression|criticis[ez]|dissent)\b", r"\barticle\s*19\b",
           r"\barticle\s*21\b", r"\bright to reputation\b", r"\bdignity\b", r"\bchilling effect\b",
           r"\bpublic (figure|personality|personalities|life|functionar(y|ies))\b",
           r"\b(public|political|fair|legitimate|healthy) criticism\b", r"\bcriticism of\b",
           r"\bpublic interest\b", r"\breasonable restrictions?\b", r"\bdemocra(cy|tic)\b",
           r"\bconstitutional (right|guarantee|freedom|value)s?\b", r"\bdecriminali[sz]"],
    "C3": [r"\b(no|lacks?|lacked|lack of|without|want of|in excess of|exceed(ed|s|ing)?( its| his| her| their| the)?|beyond( its| his| her| their| the)?|outside( its| his| her| their| the)?|oust(s|ed)?)\s+(territorial )?jurisdiction\b",
           r"\bjurisdictional (error|defect|fact|issue|question)s?\b", r"\bterritorial jurisdiction\b",
           r"\b(without|not) (holding|conducting|having held|having conducted) (an? |the |any )?(inquiry|enquiry|investigation)\b",
           r"\b(not |in)admissib", r"\blocus standi\b", r"\bfull[- ]dress trial\b",
           r"\b(opportunity|chance) (of|to be|for) (being )?heard\b", r"\bwithout (giving|affording|granting)\b[^.]{0,40}\bopportunity\b",
           r"\bremand(ed|ing)?\b", r"\bafresh\b", r"\bde novo\b",
           r"\b(should|ought) not (to )?have been (started|entertained|instituted|looked into|considered|taken into (account|consideration)|issued)\b",
           r"\b(prior |previous )?sanction\b", r"\bperson aggrieved\b", r"\baggrieved person\b",
           r"\b(barred by )?limitation\b", r"\btime[ -]barred\b", r"\bmaintainab(le|ility)\b",
           r"\birregularit(y|ies)\b", r"\bvitiat", r"\bsummons (case|trial)\b",
           r"\bwarrants? (case|trial|procedure)\b", r"\bnon[- ]?compliance\b",
           r"\bmandatory (inquiry|enquiry|requirement)\b", r"\bapplication of (judicial )?mind\b",
           r"\bmechanical (manner|order)\b", r"\bmini[- ]?trial\b", r"\b(matter|question)s? of (evidence|trial)\b",
           r"\bcan ?not be (gone|looked) into\b", r"\b(at|in) (this|the) (initial|preliminary|threshold|nascent) stage\b",
           r"\bat the stage of\b", r"\b(questions?|matters?) of fact\b",
           r"\bon the basis of (the )?evidence\b", r"\btrial (must|should|has to|shall) (go on|proceed|continue)\b",
           r"\bto be (decided|examined|seen|considered|tested) (by|at|during|in) (the )?trial\b",
           r"\bresid(es|ing) beyond\b", r"\blaches\b", r"\bundue delay\b", r"\bdelay in (filing|lodging|instituting)\b"],
    "C4": [r"\bno (cogent |reliable |credible |direct |legal |admissible |sufficient |iota of |documentary |independent )?(evidence|proof|material)s?\b",
           r"\b(insufficient|inadequate|scanty|scant|meagre) (evidence|proof|material)s?\b",
           r"\black of (evidence|proof|corroboration|material)\b", r"\bwithout (any )?(evidence|proof|corroboration)\b",
           r"\bfail(ed|s)? to (prove|establish|adduce|substantiate|produce|lead)\b",
           r"\b(not|never) (been )?(proved|proven|established)\b", r"\bcould not (prove|establish)\b",
           r"\b(un)?corroborat", r"\bbenefit of (the )?doubt\b", r"\bbeyond (a |all )?reasonable doubt\b",
           r"\bhearsay\b", r"\bno prima facie (case|material)\b",
           r"\babsence of (any )?(cogent |sufficient |reliable |legal )?(evidence|proof|material)s?\b",
           r"\bfor want of (evidence|proof|material)\b",
           r"\b(not|no|never|failed to|failure to|without)\b[^.]{0,30}\b(adduc(e|ed|ing)|lead(ing)?) (any |sufficient |cogent |reliable )?evidence\b",
           r"\bburden (of proof )?(is|lies|rests) (on|upon) (the )?(complainant|petitioner|prosecution|accused|plaintiff|respondent)\b",
           r"\bnot sufficient,? (to|for) (issue|issuing|summon|summoning|proceed|proceeding|take|taking|make|making|constitute)\b"],
    "C5": [r"\babuse of (the )?(process|power|law)\b", r"\bmisuse of (the )?(process|power|law|machinery)\b",
           r"\bmala ?fides?\b", r"\bmalafide\b", r"\bulterior (motive|purpose|object|design)s?\b",
           r"\bvexatious(ly)?\b", r"\bfrivolous\b", r"\bcounter[- ]?blast\b",
           r"\b(wreak|wreck)\w* (vengeance|revenge)\b", r"\bvengeance\b", r"\bvendetta\b",
           r"\bpersonal (grudge|animosity|rivalry|score)\b", r"\bweapon of harassment\b",
           r"\bharass(ing|ment)\b", r"\bpressuri[sz]e\b", r"\bmanifestly attended with\b"],
}

# Landmark precedents as category signals. TO CONFIRM WITH THE LAWYERS.
CITATION_PATTERNS = {
    "C1": [r"chaman lal", r"sewakram sobhani", r"balraj khanna"],
    "C2": [r"subramanian swamy", r"shreya singhal", r"rajagopal", r"khushboo", r"kaushal kishore",
           r"sakal papers"],
    "C3": [r"vijay dhanuka", r"abhijit pawar", r"udai shankar awasthi", r"birla corporation"],
    "C4": [],
    "C5": [r"bhajan lal", r"pepsi foods", r"zandu pharmaceutical", r"inder mohan goswami"],
}

SCOPE_BLOCKERS = {
    "C2": [r"\bpublic interest litigation\b"],
    "C3": [r"\binherent (jurisdiction|powers?)\b", r"\bjurisdiction under section 482\b",
           r"\b(exercise|exercising) (of )?(its |the |this |our )?(inherent |revisional |extraordinary )?jurisdiction\b",
           r"\b(supervisory|revisional|appellate|writ|extraordinary|plenary) jurisdiction\b",
           r"\bremand(ed)? (to|in) (judicial |police )?custody\b"],
    "C4": [r"\bno (evidence|proof|material)s?\b[^.]{0,40}\b(malice|mala fide|intent(ion)?|mens rea|knowledge|good faith)\b"],
    "C5": [r"\bsexual(ly)? harass"],
}

NEG = re.compile(r"\b(no|not|never|nor|neither|cannot|hardly|nothing|none|without)\b|n't\b")

def prep(text):
    t = str(text).lower()
    t = re.sub(r"\bu/s\.?\s*", "section ", t)
    t = re.sub(r"\b(s|sec|secs|section)\.?\s*(\d{3}[a-z]?)\b", r"section \2", t)
    t = re.sub(r"\bart\.?\s*(\d{2})\b", r"article \1", t)
    return re.sub(r"\s+", " ", t)

_KW = {c: [re.compile(p) for p in ps] for c, ps in KEYWORD_PATTERNS.items()}
_CI = {c: [re.compile(p) for p in ps] for c, ps in CITATION_PATTERNS.items()}
_BL = {c: [re.compile(p) for p in ps] for c, ps in SCOPE_BLOCKERS.items()}

def m1_hits(text):
    t = prep(text)
    hits = []
    for c in CATS:
        blocked = [m.span() for p in _BL.get(c, []) for m in p.finditer(t)]
        for source, pats in (("keyword", _KW[c]), ("citation", _CI[c])):
            for p in pats:
                for m in p.finditer(t):
                    s, e = m.span()
                    if any(s < be and e > bs for bs, be in blocked):
                        continue
                    negated = (source == "keyword" and not NEG.search(m.group(0))
                               and bool(NEG.search(" ".join(t[:s].split()[-NEGATION_WINDOW:]))))
                    hits.append((c, m.group(0), source, negated))
    return hits

HITS = [m1_hits(t) for t in sents.text]
M1_ALL = np.array([[int(any(h[0] == c for h in hs)) for c in CATS] for hs in HITS])
M1N_ALL = np.array([[int(any(h[0] == c and not h[3] for h in hs)) for c in CATS] for hs in HITS])
PARTY = sents.party_argument.values[:, None]
M1P_ALL = M1_ALL * (1 - PARTY)          # skip sentences restating a party's submission
for name, M in [("M1 revised keywords", M1_ALL), ("M1n + negation filter", M1N_ALL),
                ("M1p + skip party arguments", M1P_ALL)]:
    save_preds(name, M[GOLD_IDX], M)

hit_log = pd.DataFrame([(i, *h) for i, hs in enumerate(HITS) for h in hs],
                       columns=["sid", "cat", "match", "source", "negated"])
hit_log.to_csv(M1_HITS_CSV, index=False)    # used by the error review in 06
print("Corpus wide: hits per category, share from citations, share negated")
print(hit_log.groupby("cat").agg(hits=("match", "size"),
                                 citation_share=("source", lambda s: (s == "citation").mean()),
                                 negated_share=("negated", "mean")).round(2).to_string())
print("\nMost frequent matched phrases (check for generic terms):")
print(hit_log.groupby(["cat", "match"]).size().sort_values(ascending=False).groupby(level=0).head(5).to_string())

Corpus wide: hits per category, share from citations, share negated
     hits  citation_share  negated_share
cat                                     
C1   1197            0.00           0.13
C2    262            0.06           0.07
C3    446            0.00           0.19
C4    159            0.00           0.04
C5    256            0.05           0.15

Most frequent matched phrases (check for generic terms):
cat  match               
C1   exception               316
     good faith              284
     public good             190
     exceptions               95
     ingredients              75
C3   sanction                 66
C2   freedom of speech        44
C5   harassment               40
     abuse of process         37
C3   aggrieved person         34
     application of mind      33
     question of fact         31
     at the stage of          31
C2   article 19               25
C4   no material              25
C2   democratic               24
     public interest          21


## Quick check on the dev judgments

Use these scores, and the error sheet from `06_method_comparison`, to revise the phrase lists. Test scores are only reported in `06_method_comparison`.

In [4]:
pd.DataFrame({name: summarize(Y[~TEST], p["gold"][~TEST]) for name, p in load_preds().items()
              if name.startswith(("M0", "M1"))}).T.round(3)

Skipping M2 exemplar similarity: saved before the gold set changed. Re-run its notebook.
Skipping M4 hybrid M1 + M2: saved before the gold set changed. Re-run its notebook.
Skipping M6 fine-tuned, keywords then highlights: saved before the gold set changed. Re-run its notebook.
Skipping M6g fine-tuned on lawyer highlights: saved before the gold set changed. Re-run its notebook.
Skipping M6s fine-tuned on keyword labels: saved before the gold set changed. Re-run its notebook.
Skipping M6t fine-tuned, stage 2 tuned on dev: saved before the gold set changed. Re-run its notebook.


,macro_P,macro_R,macro_F1,micro_F1,exact_match_acc
M0 original keywords,0.359,0.129,0.153,0.229,0.698
M1 revised keywords,0.274,0.273,0.270,0.312,0.685
M1n + negation filter,0.287,0.252,0.262,0.300,0.702
M1p + skip party arguments,0.274,0.273,0.270,0.312,0.685
